In [ ]:
# @title Set ONLINE=False if running locally
ONLINE = True

In [ ]:
# @title Note that PySpark requires Java 8 or later with JAVA_HOME properly set.
!pip install pyspark

In [ ]:
# @title Imports
from pyspark.sql import SparkSession
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import numpy as np
import pandas as pd
import math
import os
import gdown

In [ ]:
# Download training and submission data
if ONLINE:
    url = 'https://drive.google.com/file/d/1lDEtOq644qlaw_vh0i5c3ChVfrHJiuMc/view?usp=sharing'
    output = 'data_train.csv'
    gdown.download(url, output, fuzzy=True)
    url = 'https://drive.google.com/file/d/1KMiakfzTWYr4QJU9S5DC9e5KIo_8bb24/view?usp=sharing'
    output = 'sampleSubmission.csv'
    gdown.download(url, output, fuzzy=True)

In [ ]:
# load training data
number_of_users, number_of_movies = (10000, 1000)
data_pd = pd.read_csv('data_train.csv')

# Split the dataset into train and test
train_size = 0.9
train_pd, test_pd = train_test_split(data_pd, train_size=train_size, random_state=42)

In [ ]:
def extract_users_items_predictions(data_pd):
    users, movies = \
        [np.squeeze(arr) for arr in np.split(data_pd.Id.str.extract('r(\d+)_c(\d+)').values.astype(int) - 1, 2, axis=-1)]
    predictions = data_pd.Prediction.values
    return users, movies, predictions

In [ ]:
# extract matrix data
train_users, train_movies, train_predictions = extract_users_items_predictions(train_pd)
test_users, test_movies, test_predictions = extract_users_items_predictions(test_pd)

In [ ]:
# define RMSE metric
rmse = lambda x, y: math.sqrt(mean_squared_error(x, y))

def get_score(predictions, target_values=test_predictions):
    return rmse(predictions, target_values)

In [ ]:
# @title ALS Setup
checkpoint_dir = "/tmp/spark_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

# Initialize Spark session
spark = SparkSession.builder \
    .appName("Matrix Reconstruction with ALS") \
    .config("spark.checkpoint.dir", checkpoint_dir) \
    .getOrCreate()

spark.sparkContext.setCheckpointDir(checkpoint_dir)

# Load data into a PySpark DataFrame
train_df = spark.createDataFrame([(int(u), int(m), int(p)) for u, m, p in zip(
    train_users, train_movies, train_predictions)], ["user", "item", "rating"])

In [ ]:
# @title hyperparameters
best_r = 7
best_reg = 0.08
best_i = 200

In [ ]:
# @title Training
als = ALS(rank=best_r, maxIter=best_i, regParam=best_reg)
model = als.fit(train_df)

In [ ]:
# Get user and item factors
user_factors = model.userFactors.orderBy("id").collect()
item_factors = model.itemFactors.orderBy("id").collect()

# Convert factors to NumPy arrays
user_factors_matrix = np.array([row["features"] for row in user_factors])
item_factors_matrix = np.array([row["features"] for row in item_factors])

# Reconstruct the matrix
reconstructed_matrix = np.dot(user_factors_matrix, item_factors_matrix.T)

In [ ]:
test_predictions = np.array([reconstructed_matrix[user][movie] for user, movie in zip(test_users, test_movies)]).clip(1,5)

rmse = get_score(test_predictions)
print(f'Final RMSE: {rmse}')

In [ ]:
# @title Generate predictions for ensemble
predictions_path = 'predictions'

if not os.path.exists(predictions_path):
    os.mkdir(predictions_path)

submission_pd = pd.read_csv('sampleSubmission.csv')

challenge_users, challenge_movies, _ = extract_users_items_predictions(submission_pd)
challenge_predictions = np.array([reconstructed_matrix[user][movie] for user, movie in zip(challenge_users, challenge_movies)]).clip(1, 5)

np.savez_compressed(os.path.join(predictions_path, 'als_predictions'), test=test_predictions,
                        challenge=challenge_predictions)

In [ ]:
# @title Save challenge prediction to csv
csv_path = 'csvs'
if not os.path.exists(csv_path):
    os.mkdir(csv_path)

submission_pd['Prediction'] = challenge_predictions

# Save the predictions to a new CSV file
submission_pd.to_csv(os.path.join(csv_path, 'submission_als.csv'), index=False)